# 1. File processomg 

In [9]:
# Parameters
file_cnt = 0


StatementMeta(, 2b4638ce-0775-4bb2-86aa-df0a72e35642, 13, Finished, Available, Finished, False)

## Processing the files themselves and writing to this table **dbo.ev2_file_body**

## Service functions for file processing

In [ ]:
from pyspark.sql.functions import input_file_name, regexp_extract, col, when, sum, count, substring, lit, coalesce, sequence, expr


def read_files(all_files):
    """
        Read files from the list and write to the table.
        When writing, the FILE_ID matches the id field in the file log
    """
    target_table = "dbo.ev2_file_body"
    
    print(f"Starting processing {len(all_files)} files...")

    # Create an empty DataFrame with the schema of the first file (or empty)
    # The best way is to read each file separately and add an ID to it
    combined_df = None

    for file_info in all_files:
        path = file_info["snapshot_path"]
        context_id = file_info["id"]
        
        # read the file into a temporary DataFrame
        temp_df = spark.read.json(path)
        
        # Додаємо метадані прямо в DataFrame
        temp_df = temp_df.withColumn("file_name", lit(file_info["file_name"])) \
                         .withColumn("file_id", lit(context_id))
        
        if combined_df is None:
            combined_df = temp_df
        else:
            combined_df = combined_df.unionByName(temp_df, allowMissingColumns=True)

    if combined_df:
        combined_df.createOrReplaceTempView("v_batch_incoming_data")

        # Create MERGE, adding FILE_ID
        spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING v_batch_incoming_data AS source
            ON target.TX_ID = source.TX_ID
            WHEN MATCHED THEN
                UPDATE SET 
                    target.STATUS = 'UPDATED', 
                    target.FILE_NAME = source.file_name,
                    target.FILE_ID = source.file_id
            WHEN NOT MATCHED THEN
                INSERT (TX_ID, TERMINAL_ID, OPDATE, NAME, PASSPORT, AMOUNT, CHARGE, STATUS, FILE_NAME, FILE_ID)
                VALUES (source.TX_ID, source.terminal, source.opdate, source.name, source.passport, 
                        source.amount, source.charge, 'NEW', source.file_name, source.file_id)
        """)

        print("MERGE completed successfully (with FILE_ID).")


def archive_processed_files(all_files):
    """
        Moving processed files to the archive
    """
    archive_dir = "Files/archive"
    
    # Create the archive directory if it doesn't exist
    if not mssparkutils.fs.exists(archive_dir):
        mssparkutils.fs.mkdirs(archive_dir)
        
    for file_info in all_files:
        source_path = file_info["source_path"]
        file_name = file_info["file_name"]
        
        # You can add a date to the path for better structure: archive/2026-05-04/file.json
        target_path = f"{archive_dir}/{file_name}"
        
        print(f"Moving to archive: {file_name}")
        
        # Move the original file (mv instead of cp)
        mssparkutils.fs.mv(source_path, target_path)
        
        # It is also worth deleting the temporary snapshot
        if mssparkutils.fs.exists(file_info["snapshot_path"]):
            mssparkutils.fs.rm(file_info["snapshot_path"], True)


def set_status_processed(all_files):
    """
        Update the status of files to PROCESSED
    """
    for file_info in all_files:
        file_name = file_info["file_name"]
        context_id = file_info["id"]
        # Update the status in the database for the processed file
        spark.sql("""
            UPDATE ev2_file_grn
            SET 
            processing_status = 'PROCESSED',
            processed_at = current_timestamp()
            WHERE id = :context_id AND processing_status = 'NEW'
        """, args={"context_id": context_id})
        print(f"Marking file as processed: {file_name} - ok")  

StatementMeta(, 2b4638ce-0775-4bb2-86aa-df0a72e35642, 14, Finished, Available, Finished, False)

## Directly receiving files

In [ ]:
import json
import os
result_data = {

    "status": "Success",
    "count": 0,
    "error": None    
}
#print("file_raw=" + file_raw)
counter=0

all_files=[]

try:



    print(f"Files received for processing: {file_cnt}")
    df_queue = spark.sql("SELECT subject, id FROM ev2_file_grn WHERE processing_status = 'NEW' ORDER BY processed_at ASC")
    file_list = df_queue.collect()
    if not file_list:
        print("Queue is empty. No processing needed.")
        result_data["status"] ="Success"

        mssparkutils.notebook.exit(json.dumps(result_data))

    for item in file_list:
        file_path = item["subject"]
        context_id = item["id"]        
        
        print(f"Processing file: {file_path}")
        # Here is the copy and UPDATE logic
        raw_path = file_path.strip()
        # Remove the leading slash if it gets in the way (as you noticed with "Files")
        if raw_path.startswith("/"):
            source_path = raw_path[1:]
        else:
            source_path = raw_path
        
        # Create a snapshot of the file to work with, so that we are sure it won't change during processing
        snapshots_dir = "Files/processing_snapshots"
        print("Checking if the directory exists, and creating it if not")
        if not mssparkutils.fs.exists(snapshots_dir):
            print(f"Creating directory: {snapshots_dir}")
            mssparkutils.fs.mkdirs(snapshots_dir)
        else:
            print(f"Directory {snapshots_dir} already exists")

        print("Copying file for processing")
        if source_path:
            print( "Creating a temporary copy name to avoid conflicts")
            file_name = os.path.basename(source_path)
            snapshot_path = f"{snapshots_dir}/snap_{file_name}"
            
            print(f"Copying from !{source_path}! to !{snapshot_path}!")
            # Create the snapshot
            mssparkutils.fs.cp( source_path, snapshot_path)
            
            # Now we work with snapshot_path — it is guaranteed not to change!
            print(f"Starting processing of isolated copy: {snapshot_path}")
            all_files.append( {
                    "file_name": file_name,
                    "id": context_id,
                    "source_path": source_path,
                    "snapshot_path": snapshot_path
                }
            )

        counter+=1

    print("Reading all files")  
    read_files(all_files)
    print("Reading all files - ok")   

    print(f"Marking files as processed") 
    set_status_processed(all_files)
    print(f"Marking files as processed - ok") 

    print("Starting archiving of processed files...")
    archive_processed_files(all_files)
    print("Archiving completed successfully.")

    result_data["status"]="Success"     
    result_data["count"]=counter
except Exception as e:
    print(f"Error : {e}")
    result_data["status"] = "Error"
    result_data["error"] = str(e)
  
mssparkutils.notebook.exit(json.dumps(result_data))  


StatementMeta(, 2b4638ce-0775-4bb2-86aa-df0a72e35642, 15, Finished, Available, Finished, False)

Отримано файлів для обробки: 2
Обробка файлу: /Files/terminals/19683_2026-01-19.json
Перевіряємо, чи існує папка, і створюємо її, якщо ні
каталог Files/processing_snapshots  присутній
Копіюю файл для прийому
Створюємо ім'я для тимчасової копії, щоб уникнути конфліктів
Копіюю з !Files/terminals/19683_2026-01-19.json! в !Files/processing_snapshots/snap_19683_2026-01-19.json!
Починаємо обробку ізольованої копії: Files/processing_snapshots/snap_19683_2026-01-19.json
Обробка файлу: /Files/terminals/18666_2025-05-11.json
Перевіряємо, чи існує папка, і створюємо її, якщо ні
каталог Files/processing_snapshots  присутній
Копіюю файл для прийому
Створюємо ім'я для тимчасової копії, щоб уникнути конфліктів
Копіюю з !Files/terminals/18666_2025-05-11.json! в !Files/processing_snapshots/snap_18666_2025-05-11.json!
Починаємо обробку ізольованої копії: Files/processing_snapshots/snap_18666_2025-05-11.json
Обробка файлу: /Files/terminals/14611_2025-06-23.json
Перевіряємо, чи існує папка, і створюємо її